In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [2]:


# --- CẤU HÌNH CRAWLER ---
def setup_driver():
    chrome_options = Options()
    # 1. User-Agent Desktop chuẩn (Quan trọng để không bị redirect sang Mobile)
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # 2. Tắt các cờ báo hiệu Robot
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    
    # 3. Chạy chế độ Headless (Ẩn trình duyệt) nếu muốn nhanh hơn
    # chrome_options.add_argument("--headless") 
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

# --- HÀM BÓC TÁCH DỮ LIỆU (PARSING) ---
def parse_cafef_data(html_source):
    soup = BeautifulSoup(html_source, 'html.parser')
    
    # ======================================================
    # PHẦN 1: BÓC TÁCH CHỈ SỐ CƠ BẢN (P/E, EPS, Vốn hóa...)
    # ======================================================
    print("📈 Đang xử lý: Chỉ số cơ bản...")
    basic_data = {}
    
    # Tìm khối div chứa thông tin bên phải
    right_container = soup.find('div', id='transaction-information-table-right')
    
    if right_container:
        # Lấy tất cả các dòng (items)
        items = right_container.find_all('div', class_='table-right-item')
        
        for item in items:
            # Mỗi item có 2 thẻ <p>: Tên và Giá trị
            p_tags = item.find_all('p')
            if len(p_tags) >= 2:
                # Lấy tên (xóa các thẻ span con nếu có, ví dụ "nghìn đồng")
                key_text = p_tags[0].get_text(separator=" ", strip=True)
                # Làm sạch key (bỏ phần trong ngoặc đơn nếu muốn gọn)
                # Ví dụ: "EPS cơ bản * (nghìn đồng)" -> "EPS cơ bản"
                key_clean = key_text.split('(')[0].strip().replace('*', '').strip()
                
                value = p_tags[1].text.strip()
                basic_data[key_clean] = value
    else:
        print("⚠️ Không tìm thấy bảng chỉ số bên phải (Check ID 'transaction-information-table-right')")

    df_basic = pd.DataFrame(list(basic_data.items()), columns=['Chỉ số', 'Giá trị'])

    # ======================================================
    # PHẦN 2: BÓC TÁCH BẢNG KẾT QUẢ KINH DOANH
    # ======================================================
    print("📊 Đang xử lý: Bảng Kết quả kinh doanh...")
    financial_data = []
    
    # Tìm bảng theo class
    table = soup.find('table', class_='report_detail_table')
    
    if table:
        # 1. Lấy Header để biết tên các Quý
        headers = []
        thead = table.find('thead')
        if thead:
            th_rows = thead.find_all('th')
            # Filter lấy những ô có chứa chữ "Q" hoặc năm (VD: Q3-2025)
            # Bỏ qua ô "Chỉ tiêu", "Tăng trưởng" và các ô nút bấm rỗng
            for th in th_rows:
                text = th.text.strip()
                if re.search(r'Q\d-\d{4}|Năm', text): # Regex tìm dạng Q1-2024
                    headers.append(text)
        
        # Nếu không tìm thấy header chuẩn, dùng tên mặc định
        if not headers:
            headers = ['Quý 1', 'Quý 2', 'Quý 3', 'Quý 4']

        # 2. Lấy Dữ liệu từng dòng
        tbody = table.find('tbody')
        if tbody:
            rows = tbody.find_all('tr')
            for row in rows:
                # Chỉ lấy các dòng dữ liệu (thường ko có class hoặc class btn-even/odd)
                cols = row.find_all('td')
                
                # Logic lọc rác:
                # Cấu trúc: [Tên] [ButtonRỗng] [Data1] [Data2] [Data3] [Data4] [ButtonRỗng] [Chart]
                # Ta chỉ lấy những ô có text không rỗng
                
                if len(cols) > 2:
                    # Cột 0 là Tên chỉ tiêu
                    indicator = cols[0].text.strip()
                    
                    # Các cột còn lại: Lấy text, bỏ qua ô rỗng
                    values = [c.text.strip() for c in cols[1:] if c.text.strip() != '']
                    
                    # Ghép dữ liệu vào
                    row_dict = {'Chỉ tiêu': indicator}
                    
                    # Map giá trị vào Header tương ứng
                    # Lấy tối đa số lượng header tìm được
                    for i, h in enumerate(headers):
                        if i < len(values):
                            row_dict[h] = values[i]
                            
                    financial_data.append(row_dict)
    else:
         print("⚠️ Không tìm thấy bảng tài chính (Check class 'report_detail_table')")

    df_finance = pd.DataFrame(financial_data)
    
    return df_basic, df_finance

# --- HÀM CHÍNH (MAIN FLOW) ---
def crawl_full_cafef(symbol):
    # URL Desktop chuẩn
    url = f"https://cafef.vn/du-lieu/hose/{symbol}-cong-ty.chn" 
    # Lưu ý: CafeF có quy tắc URL hơi lằng nhằng, đôi khi cần slug tên công ty.
    # Nhưng thường link này sẽ tự redirect đúng:
    # https://s.cafef.vn/hose/HRC-cong-ty-co-phan-cao-su-hoa-binh.chn
    
    print(f"🚀 Bắt đầu crawl mã: {symbol}")
    driver = setup_driver()
    
    try:
        driver.get(url)
        print("⏳ Đang đợi trang web tải (8 giây)...")
        time.sleep(8) # Đợi JavaScript render xong
        
        # Có thể scroll xuống một chút để kích hoạt lazy loading (nếu có)
        driver.execute_script("window.scrollTo(0, 500);")
        time.sleep(2)
        
        print("📥 Đang lấy HTML source...")
        html_source = driver.page_source
        
        # Gọi hàm parsing
        df_basic, df_fin = parse_cafef_data(html_source)
        
        return df_basic, df_fin
        
    except Exception as e:
        print(f"❌ Lỗi: {e}")
        return None, None
    finally:
        driver.quit()
        print("✅ Đã đóng trình duyệt.")



In [3]:
# --- CHẠY THỬ ---
if __name__ == "__main__":
    ticker = "HRC" # Thử mã HRC
    df_info, df_bctc = crawl_full_cafef(ticker)
    
    if df_info is not None and not df_info.empty:
        print("\n=== 1. CHỈ SỐ CƠ BẢN ===")
        print(df_info)
        # Xuất Excel
        # df_info.to_excel(f"{ticker}_basic.xlsx", index=False)
        
    if df_bctc is not None and not df_bctc.empty:
        print("\n=== 2. KẾT QUẢ KINH DOANH (Gần nhất) ===")
        print(df_bctc.head(10)) # In 10 dòng đầu
        # Xuất Excel
        # df_bctc.to_excel(f"{ticker}_financial.xlsx", index=False)

🚀 Bắt đầu crawl mã: HRC
⏳ Đang đợi trang web tải (8 giây)...
📥 Đang lấy HTML source...
📈 Đang xử lý: Chỉ số cơ bản...
📊 Đang xử lý: Bảng Kết quả kinh doanh...
✅ Đã đóng trình duyệt.

=== 1. CHỈ SỐ CƠ BẢN ===
                       Chỉ số     Giá trị
0                  EPS cơ bản        0.93
1               EPS pha loãng        0.93
2                         P/E       30.45
3         Giá trị sổ sách/ cp    20,525.1
4                         P/B        1.39
5          Vốn hóa thị trường      859.38
6  KLGD khớp lệnh TB 10 phiên       1,480
7          KLCP đang niêm yết  30,206,622
8               KLCP lưu hành  30,206,622

=== 2. KẾT QUẢ KINH DOANH (Gần nhất) ===
                            Chỉ tiêu     Quý 1     Quý 2     Quý 3     Quý 4
0         Doanh thu bán hàng và CCDV  37.35 tỷ  37.35 tỷ  59.46 tỷ  67.23 tỷ
1                   Giá vốn hàng bán  29.35 tỷ  29.35 tỷ   54.2 tỷ  56.15 tỷ
2        Lợi nhuận gộp về BH và CCDV      8 tỷ      8 tỷ   5.26 tỷ  11.08 tỷ
3                Lợi n

In [4]:
# --- CHẠY THỬ ---
if __name__ == "__main__":
    ticker = "VHM" # Thử mã HRC
    df_info, df_bctc = crawl_full_cafef(ticker)
    
    if df_info is not None and not df_info.empty:
        print("\n=== 1. CHỈ SỐ CƠ BẢN ===")
        print(df_info)
        # Xuất Excel
        # df_info.to_excel(f"{ticker}_basic.xlsx", index=False)
        
    if df_bctc is not None and not df_bctc.empty:
        print("\n=== 2. KẾT QUẢ KINH DOANH (Gần nhất) ===")
        print(df_bctc.head(10)) # In 10 dòng đầu
        # Xuất Excel
        # df_bctc.to_excel(f"{ticker}_financial.xlsx", index=False)

🚀 Bắt đầu crawl mã: VHM
⏳ Đang đợi trang web tải (8 giây)...
📥 Đang lấy HTML source...
📈 Đang xử lý: Chỉ số cơ bản...
📊 Đang xử lý: Bảng Kết quả kinh doanh...
✅ Đã đóng trình duyệt.

=== 1. CHỈ SỐ CƠ BẢN ===
                       Chỉ số        Giá trị
0                  EPS cơ bản           5.96
1               EPS pha loãng           5.96
2                         P/E          20.81
3         Giá trị sổ sách/ cp      49,903.72
4                         P/B           2.17
5          Vốn hóa thị trường     509,729.83
6  KLGD khớp lệnh TB 10 phiên     10,783,880
7          KLCP đang niêm yết  4,107,412,004
8               KLCP lưu hành  4,107,412,004

=== 2. KẾT QUẢ KINH DOANH (Gần nhất) ===
                            Chỉ tiêu         Quý 1         Quý 2  \
0         Doanh thu bán hàng và CCDV  33,136.43 tỷ  15,697.92 tỷ   
1                   Giá vốn hàng bán  21,179.87 tỷ  10,539.99 tỷ   
2        Lợi nhuận gộp về BH và CCDV  11,956.56 tỷ   5,157.93 tỷ   
3                Lợi nhuận t